# S0 — 02.02: Multiple Linear Regression: From Features to Coefficients

**Module:** 02 — Supervised Regression  
**Prerequisites:** 01.01–01.06 and 02.01  
**Difficulty:** introductory  
**Estimated time:** 45–60 minutes

> **Learner question:** How does least squares use several features at once to choose an intercept and one coefficient per feature?

This is a deterministic, synthetic, dimensionless demonstration. Its displayed losses describe this small table only; they are not performance or generalisation evidence.

## S1 — From one feature to many

The previous lesson used `y_hat = b_0 + b_1*x`. With two input features, the model has one shared intercept and one coefficient per feature:

`y_hat_i = b_0 + b_1*x_i1 + b_2*x_i2`.

Multiple linear regression means multiple input features, not multiple target columns.

## S2 — Rows, columns, and targets

Each row is one observation. The two columns are the two features, and `y` supplies one target per row.

In [ ]:
import numpy as np

RTOL = 0.0
ATOL = 1e-12
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [1.0, 2.0]], dtype=np.float64)
y = np.array([5.0, 1.0, 4.0, 8.0, 3.0], dtype=np.float64)
X_new = np.array([[3.0, 1.0], [0.0, 2.0], [2.0, 2.0]], dtype=np.float64)


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_new shape:", X_new.shape)


## S3 — Prediction rule and shapes

Let `beta = [b_1, b_2]`, with shape `(2,)`. A row prediction is a dot product plus the intercept:

`y_hat = b_0 + X @ beta`.

After fitting, the prediction vector has shape `(5,)`, matching `y`.

## S4 — Residuals, RSS, and MSE

The residual remains observed minus fitted: `r_i = y_i - y_hat_i`. Ordinary least squares minimizes `RSS = sum(r_i**2)`. Here `MSE = RSS / 5`; neither is a held-out score.

## S5 — Center the problem

Center each feature column and the target. This separates the intercept from the feature coefficients.

In [ ]:
x_bar = np.mean(X, axis=0, dtype=np.float64)
y_bar = np.float64(np.mean(y, dtype=np.float64))
Z = X - x_bar
y_c = y - y_bar
print("x_bar:", x_bar)
print("y_bar:", y_bar)
print("Z:\n", Z)
print("y_c:", y_c)
print("Z.T @ Z:\n", Z.T @ Z)
print("Z.T @ y_c:", Z.T @ y_c)


## S6 — The two-feature least-squares system

The centered equations are `(Z.T @ Z) @ beta = Z.T @ y_c`. For this table, the left matrix is `[[2, 0], [0, 2]]` and the right side is `[7, -2]`, so `beta = [3.5, -1.0]`. Then `b_0 = y_bar - x_bar @ beta = 1.7`.

In [ ]:
beta = np.linalg.solve(Z.T @ Z, Z.T @ y_c).astype(np.float64)
b_0 = np.float64(y_bar - x_bar @ beta)
print("beta [b_1, b_2]:", beta)
print("b_0:", b_0)


## S7 — Complete hand calculation

The fitted rule is `y_hat_i = 1.7 + 3.5*x_i1 - 1.0*x_i2`. It gives training predictions `[5.2, 0.7, 4.2, 7.7, 3.2]` and residuals `[-0.2, 0.3, -0.2, 0.3, -0.2]`.

`RSS = 0.30` and `MSE = 0.06`. For `X_new`, the predictions are `[11.2, -0.3, 6.7]`.

In [ ]:
y_hat = np.asarray(b_0 + X @ beta, dtype=np.float64)
residuals = np.asarray(y - y_hat, dtype=np.float64)
rss = np.float64(np.sum(residuals**2, dtype=np.float64))
mse = np.float64(rss / y.size)
new_predictions = np.asarray(b_0 + X_new @ beta, dtype=np.float64)
print("y_hat:", y_hat)
print("residuals:", residuals)
print("RSS:", rss, "MSE:", mse)
print("new predictions:", new_predictions)


## S8 — Deterministic NumPy implementation

The implementation contract requires NumPy arrays with exact `np.float64` dtype, two-dimensional `X` and `X_new`, one-dimensional `y`, non-empty finite values, matching sample counts, compatible feature counts, and finite scalar values. These checks make shape mistakes visible instead of silently coercing inputs.

## S9 — Verification and library comparison

First compare the calculation with literal expected values. Then use `LinearRegression(fit_intercept=True)` as an independent library check. The library call confirms the result; it is not the derivation.

In [ ]:
np.testing.assert_allclose(beta, [3.5, -1.0], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(b_0, 1.7, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(y_hat, [5.2, 0.7, 4.2, 7.7, 3.2], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(residuals, [-0.2, 0.3, -0.2, 0.3, -0.2], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose([rss, mse], [0.3, 0.06], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(new_predictions, [11.2, -0.3, 6.7], rtol=RTOL, atol=ATOL)

from sklearn.linear_model import LinearRegression
estimator = LinearRegression(fit_intercept=True).fit(X, y)
np.testing.assert_allclose(estimator.coef_, beta, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(estimator.intercept_, b_0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(estimator.predict(X_new), new_predictions, rtol=RTOL, atol=ATOL)
print("All hand-calculation and library checks passed.")


## S10 — Static visual

![Multiple linear regression visual](assets/multiple_linear_regression.png)

The circles are observed targets, squares are fitted targets, and dashed segments are residuals labelled observed minus fitted. The table keeps the two feature columns visible. The values are synthetic and dimensionless; the visual is not evidence of performance or generalisation.

## S11 — Failure modes and exercises

Watch for a one-dimensional `X`, a wrong feature count in `X_new`, mismatched target rows, non-finite values, and non-`float64` arrays. Correlated feature columns can make coefficients sensitive; this lesson only flags that issue. Coefficients describe this fitted equation and are not causal effects.

1. Identify the shapes of `X`, `y`, `beta`, and `X_new`.
2. Compute the prediction for `[2.0, 2.0]` by hand.
3. Explain why residual `0.3` means observed target is above its fitted value.
4. State one validation error caused by changing `X_new` to shape `(3, 1)`.

## S12 — Takeaways, references, and continuity

Multiple linear regression keeps the OLS objective from 02.01 but replaces one slope with a coefficient vector: one coefficient for each feature and one shared intercept. Matrix shapes make the row-wise calculation explicit.

References: [scikit-learn 1.5 ordinary least squares](https://scikit-learn.org/1.5/modules/linear_model.html#ordinary-least-squares), [`LinearRegression`](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html), and [NumPy `assert_allclose`](https://numpy.org/doc/1.26/reference/generated/numpy.testing.assert_allclose.html).

The next lesson should preserve this least-squares foundation while moving to the next production-board regression topic. This lesson does not teach regularization, evaluation, inference, causality, or deployment.